# Shader Generation Quality Evaluation

**Project:** Vibe-Synth  
**Purpose:** Evaluate the quality of WGSL and GLSL fragment shaders generated by the LLM pipeline  
**Metrics:** Validation pass rate, self-correction rate, parameter manifest completeness, template match confidence, platform coverage  

Run all cells top-to-bottom. Requires the backend at `http://localhost:8000`.

In [ ]:
# ---------------------------------------------------------------------------
# Dependencies
# ---------------------------------------------------------------------------
import json
import time
import asyncio
import re
import httpx
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

API_BASE = "http://localhost:8000/api/v1"
print("Dependencies loaded.")

## 1. Test Vibe Set

Shader Vibes covering both WebGPU (WGSL) and WebGL (GLSL) targets, across colour, distortion, blur, grain, and time-animated effects.

In [ ]:
TEST_VIBES = [
    # WebGPU / WGSL
    {"id": "sh_01", "vibe": "Frosted glass distortion with cyberpunk chromatic aberration",
     "platform": "webgpu", "expected_technique": "uv_distortion+rgb_split"},
    {"id": "sh_02", "vibe": "Glitchy VHS scanline tears with horizontal RGB shift",
     "platform": "webgpu", "expected_technique": "glitch"},
    {"id": "sh_03", "vibe": "Soft dreamy Gaussian blur — gentle out-of-focus haze",
     "platform": "webgpu", "expected_technique": "blur"},
    {"id": "sh_04", "vibe": "Pencil sketch Sobel edge detection with dark outlines",
     "platform": "webgpu", "expected_technique": "edge_detection"},
    {"id": "sh_05", "vibe": "Rippling water surface — animated sine wave UV distortion",
     "platform": "webgpu", "expected_technique": "uv_distortion+animation"},
    {"id": "sh_06", "vibe": "Dark cinema vignette — heavy radial darkening toward edges",
     "platform": "webgpu", "expected_technique": "vignette"},
    {"id": "sh_07", "vibe": "Pixelate mosaic — blocky retro 8-bit pixel art look",
     "platform": "webgpu", "expected_technique": "pixelate"},

    # WebGL / GLSL
    {"id": "sh_08", "vibe": "Warm vintage sepia film grain like a Super 8 home movie",
     "platform": "webgl", "expected_technique": "color_grade+grain"},
    {"id": "sh_09", "vibe": "Cold desaturated blue shift — icy winter morning colour grade",
     "platform": "webgl", "expected_technique": "color_grade"},
    {"id": "sh_10", "vibe": "Neon cyberpunk colour boost with strong saturation and glow",
     "platform": "webgl", "expected_technique": "color_grade+bloom"},

    # Edge cases
    {"id": "sh_11", "vibe": "Do nothing — clean passthrough, output the input unchanged",
     "platform": "webgpu", "expected_technique": "passthrough"},
    {"id": "sh_12", "vibe": "Extremely subtle barely-visible film grain with no other changes",
     "platform": "webgl", "expected_technique": "grain"},
]

print(f"{len(TEST_VIBES)} shader test Vibes loaded.")

## 2. Run Generation Pipeline

In [ ]:
async def generate_shader(vibe: str, platform: str, force_refresh: bool = True) -> dict:
    """Call POST /api/v1/generate/shader and return the full response dict."""
    async with httpx.AsyncClient(timeout=60.0) as client:
        resp = await client.post(
            f"{API_BASE}/generate/shader",
            json={
                "prompt":          vibe,
                "input_type":      "video_stream",
                "target_platform": platform,
                "max_parameters":  8,
                "force_refresh":   force_refresh,
            },
        )
        resp.raise_for_status()
        return resp.json()


results = []

for test in TEST_VIBES:
    print(f"Running {test['id']} [{test['platform']}]: {test['vibe'][:55]}…", end=" ", flush=True)
    t0 = time.perf_counter()
    try:
        response = asyncio.run(generate_shader(test["vibe"], test["platform"]))
        elapsed  = round((time.perf_counter() - t0) * 1000)
        results.append({
            "id":               test["id"],
            "platform":         test["platform"],
            "expected_technique": test["expected_technique"],
            "vibe":             test["vibe"],
            "status":           response.get("status"),
            "compile_time_ms":  response.get("compile_time_ms"),
            "cache_hit":        response.get("cache_hit"),
            "param_count":      len(response.get("parameters", [])),
            "corrections":      response.get("self_correction_attempts", 0),
            "shader_len":       len(response.get("shader_code", "")),
            "wall_time_ms":     elapsed,
            "error":            None,
        })
        print(f"OK ({elapsed} ms)")
    except Exception as exc:
        elapsed = round((time.perf_counter() - t0) * 1000)
        results.append({
            "id":               test["id"],
            "platform":         test["platform"],
            "expected_technique": test["expected_technique"],
            "vibe":             test["vibe"],
            "status":           "error",
            "error":            str(exc),
            "wall_time_ms":     elapsed,
        })
        print(f"ERROR: {exc}")

df = pd.DataFrame(results)
print(f"\n{len(df)} tests completed.")

## 3. Summary Statistics

In [ ]:
success_mask    = df["status"].isin(["success", "cache_hit", "self_corrected"])
success_rate    = success_mask.mean() * 100
correction_rate = (df.loc[success_mask, "corrections"] > 0).mean() * 100
avg_compile     = df.loc[success_mask, "compile_time_ms"].mean()
avg_params      = df.loc[success_mask, "param_count"].mean()
zero_params     = (df.loc[success_mask, "param_count"] == 0).sum()

wgsl_mask = df["platform"] == "webgpu"
glsl_mask = df["platform"] == "webgl"
wgsl_success = (df.loc[wgsl_mask, "status"].isin(["success", "cache_hit", "self_corrected"])).mean() * 100
glsl_success = (df.loc[glsl_mask, "status"].isin(["success", "cache_hit", "self_corrected"])).mean() * 100

print(f"{'Metric':<40} {'Value':>10}")
print("-" * 52)
print(f"{'Overall success rate':<40} {success_rate:>9.1f}%")
print(f"{'WGSL (WebGPU) success rate':<40} {wgsl_success:>9.1f}%")
print(f"{'GLSL (WebGL) success rate':<40} {glsl_success:>9.1f}%")
print(f"{'Self-correction rate (of successes)':<40} {correction_rate:>9.1f}%")
print(f"{'Avg compile time (ms)':<40} {avg_compile:>10.0f}")
print(f"{'Avg parameter count':<40} {avg_params:>10.1f}")
print(f"{'Generations with 0 parameters':<40} {zero_params:>10}")

## 4. Results Table

In [ ]:
display_cols = [
    "id", "platform", "expected_technique", "status",
    "compile_time_ms", "param_count", "corrections", "shader_len",
]
display(df[display_cols].style.applymap(
    lambda v: "background-color: #ffcccc" if v == "error" else (
        "background-color: #fff3cd" if v == "self_corrected" else ""
    ),
    subset=["status"],
))

## 5. Platform Comparison Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Success rate by platform
platform_success = df.groupby("platform")["status"].apply(
    lambda s: s.isin(["success", "cache_hit", "self_corrected"]).mean() * 100
)
platform_success.plot(kind="bar", ax=axes[0], color=["steelblue", "teal"], edgecolor="white")
axes[0].set_title("Success Rate by Platform")
axes[0].set_ylabel("Success rate (%)")
axes[0].set_ylim(0, 110)
axes[0].tick_params(axis="x", rotation=0)

# Compile time by platform
df.loc[success_mask].boxplot(
    column="compile_time_ms", by="platform", ax=axes[1],
    patch_artist=True,
    boxprops=dict(facecolor="lightsteelblue"),
)
axes[1].set_title("Compile Time by Platform")
axes[1].set_ylabel("Compile time (ms)")
axes[1].set_xlabel("")

# Parameter count distribution
param_counts = df.loc[success_mask, "param_count"].value_counts().sort_index()
param_counts.plot(kind="bar", ax=axes[2], color="mediumseagreen", edgecolor="white")
axes[2].set_title("Parameter Count Distribution")
axes[2].set_xlabel("Parameter count")
axes[2].set_ylabel("Frequency")
axes[2].tick_params(axis="x", rotation=0)

plt.suptitle("")
plt.tight_layout()
plt.savefig("shader_quality_plots.png", dpi=150)
plt.show()
print("Plot saved to shader_quality_plots.png")

## 6. Static Validation Replay

Re-run the Validator on every generated shader code string to confirm the static checks pass.

In [ ]:
import sys
sys.path.insert(0, "..")

from app.config import get_settings
from app.services.validator import Validator

settings  = get_settings()
validator = Validator(settings)
val_results = []

for row in results:
    code = row.get("shader_code", "")
    if not code:
        val_results.append({"id": row["id"], "passed": None, "errors": ["no code"]})
        continue
    platform = row.get("platform", "webgpu")
    # Wrap in fenced block for the validator
    lang  = "wgsl" if platform == "webgpu" else "glsl"
    raw   = f"```{lang}\n{code}\n```\n```json-params\n[]\n```"
    vr    = validator.validate_shader(raw_output=raw, target_platform=platform)
    val_results.append({
        "id":      row["id"],
        "passed":  vr.passed,
        "errors":  vr.errors,
    })

val_df = pd.DataFrame(val_results)
passed_count = val_df["passed"].sum()
print(f"Validation passed: {passed_count}/{len(val_df)}")
display(val_df[["id", "passed", "errors"]])

## 7. Loop Safety Audit

Check that no generated shader contains dynamic loop bounds (a common LLM mistake).

In [ ]:
DYNAMIC_LOOP_PATTERN = re.compile(
    r"for\s*\([^)]*;[^;]*;[^)]*\)\s*\{[^}]*\b(?!\d)\w+\b",
    re.MULTILINE,
)

loop_issues = []
for row in results:
    code = row.get("shader_code", "")
    if not code:
        continue
    # Simple heuristic: look for 'for' loops whose bound is a uniform variable
    if re.search(r"for\s*\(\s*int\s+\w+\s*=\s*0\s*;[^;]*<\s*u_", code):
        loop_issues.append({"id": row["id"], "issue": "Dynamic loop bound references uniform"})
    elif re.search(r"while\s*\(", code):
        loop_issues.append({"id": row["id"], "issue": "while loop detected"})

if loop_issues:
    print(f"WARNING: {len(loop_issues)} loop safety issue(s) found:")
    display(pd.DataFrame(loop_issues))
else:
    print("All shaders passed loop safety audit — no dynamic bounds or while loops.")

## 8. Export Results

In [ ]:
out_path = Path("eval_shader_results.csv")
df.to_csv(out_path, index=False)
print(f"Results saved to {out_path} ({len(df)} rows).")